In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import platform

import torch
from google.colab import drive

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/FacialVisualProfile")
assert root.is_dir(), root

print("Python:", platform.python_version())
for package in ("torch", "torchvision", "numpy", "Pillow", "scikit-learn"):
    try:
        print(f"{package}:", importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Oracle 优先复用已验证的 M3 训练源码快照。
source = root / "runs/m3_seed0/source"
print("\nTraining source snapshot:", source)

for filename in (
    "models/resnet18_geometry.py",
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "configs/attributes.yaml",
    "evaluation/attributes.py",
):
    path = source / filename
    print(f"\n{filename}: exists={path.is_file()}")
    if path.is_file():
        print("SHA256:", hashlib.sha256(path.read_bytes()).hexdigest())
        if filename == "datasets/celeba.py":
            print(path.read_text(encoding="utf-8"))

print("\nDrive data entries:")
data = root / "data"
if data.is_dir():
    for path in sorted(data.iterdir()):
        print(path.name, "directory" if path.is_dir() else "file")
else:
    print("Data directory not found:", data)

print("\nExisting Oracle run directories:")
runs = root / "runs"
if runs.is_dir():
    for path in sorted(runs.iterdir()):
        if "oracle" in path.name.lower():
            print(path)

print("\nDay 18 environment and source inspection: COMPLETE")

Mounted at /content/drive
Python: 3.13.15
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
numpy: 2.1.3
Pillow: 11.3.0
scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4

Training source snapshot: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/source

models/resnet18_geometry.py: exists=True
SHA256: 14786dc26945214a856495a367c7adab45eb4dd9ff486fd8a12720fec0fb501b

datasets/celeba.py: exists=True
SHA256: 1d599c0e074a75f8a9bae645989681e7f93a3a3c8f8566b9530f8a69c34ddb02
"""CelebA images, 24 selected attributes, and aligned GT landmarks."""
from pathlib import Path

from PIL import Image
import torch
from torch.utils.data import Dataset
import yaml

from datasets.preprocessing import prepare


"""
when we inherit Dataset and want to create our own dataset
we need to override __len__ and __getitem__
"""

"""
information about the 24 attributes is stored in attrubutes.yaml
"""

class CelebAAttributes(Dataset):

    # when creating an instance:
    # we need to specify which datase

Getting the training data

In [2]:
from pathlib import Path
import hashlib
import importlib
import os
import shutil
import sys
import zipfile

import numpy as np
import torch
from PIL import Image

root = Path("/content/drive/MyDrive/FacialVisualProfile")
source = root / "runs/m3_seed0/source"
workspace = Path("/content/day18_oracle_workspace")
data_root = Path("/content/day18_oracle_data/raw")
image_dir = data_root / "img_align_celeba"

expected_hashes = {
    "models/resnet18_geometry.py":
        "14786dc26945214a856495a367c7adab45eb4dd9ff486fd8a12720fec0fb501b",
    "datasets/celeba.py":
        "1d599c0e074a75f8a9bae645989681e7f93a3a3c8f8566b9530f8a69c34ddb02",
    "datasets/preprocessing.py":
        "895e3e93641878068072043ff1dfd16a5fb32adcd270617a2890694962fd966f",
    "configs/attributes.yaml":
        "33acae803c307686d45b6fd2c5d64749074de8e81481c533f20fede416d0528b",
    "evaluation/attributes.py":
        "d45ebd6e28b94dad0af46547f8da04130f0f193289f4684b1c8540e46f1026bd",
}

# 恢复训练时使用的源码，保留原始 comments。
for filename, expected in expected_hashes.items():
    original = source / filename
    data = original.read_bytes()
    assert hashlib.sha256(data).hexdigest() == expected, filename
    destination = workspace / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == data, filename
    else:
        destination.write_bytes(data)

for package in ("datasets", "models", "evaluation"):
    init = workspace / package / "__init__.py"
    if not init.exists():
        init.write_text("", encoding="utf-8")

data_root.mkdir(parents=True, exist_ok=True)
image_dir.mkdir(parents=True, exist_ok=True)

for filename in (
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
):
    original = root / "data" / filename
    destination = data_root / filename
    if destination.exists():
        assert destination.read_bytes() == original.read_bytes(), filename
    else:
        shutil.copy2(original, destination)

# 根据官方 partition，只选择 train 和 validation 图片。
membership = {}
for line in (data_root / "list_eval_partition.txt").read_text().splitlines():
    name, code = line.split()
    assert name not in membership
    assert code in ("0", "1", "2")
    membership[name] = code

counts = {
    split: sum(code == split for code in membership.values())
    for split in ("0", "1", "2")
}
assert counts == {"0": 162770, "1": 19867, "2": 19962}

required = sorted(
    name for name, code in membership.items() if code in ("0", "1")
)
assert len(required) == 182637
missing = [name for name in required if not (image_dir / name).is_file()]

if missing:
    local_zip = Path("/content/day18_img_align_celeba.zip")
    original_zip = root / "data/img_align_celeba.zip"
    if not local_zip.exists():
        print("Copying image ZIP from Drive...")
        temporary = local_zip.with_suffix(".zip.tmp")
        shutil.copy2(original_zip, temporary)
        temporary.replace(local_zip)
    assert local_zip.stat().st_size == original_zip.stat().st_size

    print("Images to extract:", len(missing))
    with zipfile.ZipFile(local_zip) as archive:
        members = {}
        for member in archive.infolist():
            if member.is_dir():
                continue
            name = Path(member.filename).name
            if name in membership:
                assert name not in members, f"Duplicate ZIP image: {name}"
                members[name] = member

        assert set(required).issubset(members)

        for index, name in enumerate(missing, 1):
            destination = image_dir / name
            temporary = destination.with_suffix(".jpg.tmp")
            with archive.open(members[name]) as reader:
                with temporary.open("wb") as writer:
                    shutil.copyfileobj(reader, writer)
            temporary.replace(destination)

            if index % 20000 == 0 or index == len(missing):
                print(f"Extracted {index}/{len(missing)}")

assert all((image_dir / name).is_file() for name in required)
extracted = {path.name for path in image_dir.glob("*.jpg")}
assert extracted == set(required), "Unexpected or missing images"
print("Train/validation image membership: PASS")

# 使用隔离工作目录；拒绝误用此前导入的其他 datasets/models 包。
os.chdir(workspace)
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))

for package in ("datasets", "models"):
    if package in sys.modules:
        locations = list(getattr(sys.modules[package], "__path__", []))
        assert str(workspace / package) in locations, (
            f"{package} already loaded from another location; "
            "restart runtime and rerun the Day 18 cells"
        )

importlib.invalidate_caches()
from datasets.celeba import CelebAAttributes
from models.resnet18_geometry import ResNet18GeometryAttributes

train_dataset = CelebAAttributes(split="train", root=data_root)
valid_dataset = CelebAAttributes(split="valid", root=data_root)

assert len(train_dataset) == 162770
assert len(valid_dataset) == 19867
assert train_dataset.attribute_names == valid_dataset.attribute_names
assert len(train_dataset.attribute_names) == 24

for dataset, code in ((train_dataset, "0"), (valid_dataset, "1")):
    expected_names = {
        name for name, split in membership.items() if split == code
    }
    assert len(dataset.names) == len(set(dataset.names))
    assert set(dataset.names) == expected_names

# 包含历史 MTCNN no_face 样本 000199.jpg。
# Oracle 使用 GT，因此不依赖该图片的检测成功状态。
selected = [
    (train_dataset, "000001.jpg"),
    (train_dataset, "000199.jpg"),
    (valid_dataset, "162771.jpg"),
    (valid_dataset, "182637.jpg"),
]
samples = []

for dataset, name in selected:
    image_tensor, target, points, returned_name = dataset[
        dataset.names.index(name)
    ]
    assert returned_name == name
    assert image_tensor.shape == (3, 224, 224)
    assert target.shape == (24,)
    assert points.shape == (10,)
    assert torch.isfinite(image_tensor).all()
    assert torch.isfinite(points).all()
    assert torch.isin(target, torch.tensor([0.0, 1.0])).all()

    # 独立计算 GT 映射，确认坐标确实来自标注。
    with Image.open(image_dir / name) as image:
        width, height = image.size
    scale = 224 / max(width, height)
    new_width = round(width * scale)
    new_height = round(height * scale)
    left = (224 - new_width) // 2
    top = (224 - new_height) // 2

    expected = dataset.landmarks[name].clone().reshape(5, 2).float()
    expected[:, 0] = (
        (expected[:, 0] + .5) * new_width / width - .5 + left
    )
    expected[:, 1] = (
        (expected[:, 1] + .5) * new_height / height - .5 + top
    )
    expected = (2 * expected / 223 - 1).flatten()
    torch.testing.assert_close(points, expected, rtol=0, atol=0)
    samples.append((image_tensor, target, points))

images = torch.stack([sample[0] for sample in samples])
targets = torch.stack([sample[1] for sample in samples])
gt_points = torch.stack([sample[2] for sample in samples])
geometry_valid = torch.ones(len(samples), dtype=torch.bool)

# 随机权重仅用于接线检查；正式训练必须另建 pretrained=True 模型。
torch.manual_seed(0)
smoke_model = ResNet18GeometryAttributes(
    num_attributes=24, pretrained=False
).eval()

with torch.inference_mode():
    logits = smoke_model(images, gt_points, geometry_valid)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, targets
    )

assert logits.shape == (4, 24)
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

del smoke_model

print("Dataset sizes:", len(train_dataset), len(valid_dataset))
print("GT coordinate mapping: PASS")
print("Historical no_face sample retained with GT: PASS")
print("Oracle smoke logits:", tuple(logits.shape))
print("Untrained smoke BCE:", float(loss))
print("Workspace:", workspace)
print("Data root:", data_root)
print("No optimizer updates performed")
print("Day 18 Oracle data and forward preparation: PASS")

Copying image ZIP from Drive...
Images to extract: 182637
Extracted 20000/182637
Extracted 40000/182637
Extracted 60000/182637
Extracted 80000/182637
Extracted 100000/182637
Extracted 120000/182637
Extracted 140000/182637
Extracted 160000/182637
Extracted 180000/182637
Extracted 182637/182637
Train/validation image membership: PASS
Dataset sizes: 162770 19867
GT coordinate mapping: PASS
Historical no_face sample retained with GT: PASS
Oracle smoke logits: (4, 24)
Untrained smoke BCE: 0.9106609225273132
Workspace: /content/day18_oracle_workspace
Data root: /content/day18_oracle_data/raw
No optimizer updates performed
Day 18 Oracle data and forward preparation: PASS


In [3]:
from pathlib import Path
import inspect
import json
import torch

root = Path("/content/drive/MyDrive/FacialVisualProfile")
m3_run = root / "runs/m3_seed0"
checkpoint_path = m3_run / "epoch_02.pt"

assert checkpoint_path.is_file(), checkpoint_path

# 讀取自己保存的 M3 checkpoint，只輸出設定，不輸出權重。
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint keys:", sorted(checkpoint.keys()))

for key in ("epoch", "config", "best_epoch", "best_map"):
    if key in checkpoint:
        print(f"\nCheckpoint {key}:")
        print(json.dumps(
            checkpoint[key],
            indent=2,
            ensure_ascii=False,
            default=str,
        ))

config_path = m3_run / "config.json"
if config_path.is_file():
    print("\nSaved run config.json:")
    print(config_path.read_text(encoding="utf-8"))

from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

print("\nModel initialization:")
print(inspect.getsource(ResNet18GeometryAttributes.__init__))

print("\nEvaluation interface:")
print(inspect.signature(evaluate_attributes))
print(inspect.getsource(evaluate_attributes))

del checkpoint
print("\nDay 18 historical training configuration inspection: COMPLETE")

Checkpoint keys: ['best', 'config', 'epoch', 'history', 'model_state_dict', 'optimizer_state_dict']

Checkpoint epoch:
2

Checkpoint config:
{
  "model": "ResNet18GeometryAttributes",
  "seed": 0,
  "epochs": 10,
  "batch_size": 64,
  "num_workers": 2,
  "optimizer": "Adam",
  "learning_rate": 0.0001,
  "weight_decay": 0.0,
  "precision": "FP32",
  "initial_weights": "ResNet18_Weights.IMAGENET1K_V1",
  "augmentation": "None; use existing shared preprocessing",
  "f1_threshold": 0.5,
  "selection_metric": "validation_mAP",
  "selection_tie_rule": "earliest epoch",
  "attribute_names": [
    "Smiling",
    "Mouth_Slightly_Open",
    "Eyeglasses",
    "Bangs",
    "Wearing_Hat",
    "Mustache",
    "No_Beard",
    "Black_Hair",
    "Bushy_Eyebrows",
    "Arched_Eyebrows",
    "High_Cheekbones",
    "Bags_Under_Eyes",
    "Big_Lips",
    "Big_Nose",
    "Double_Chin",
    "Oval_Face",
    "Receding_Hairline",
    "Blond_Hair",
    "Brown_Hair",
    "Gray_Hair",
    "Bald",
    "Straight_Ha

offcial training baby ~

In [4]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

root = Path("/content/drive/MyDrive/FacialVisualProfile")
workspace = Path("/content/day18_oracle_workspace")
run_dir = root / "runs/oracle_seed0"
device = torch.device("cuda")

assert torch.cuda.is_available(), "Enable a GPU runtime first"
assert len(train_dataset) == 162770
assert len(valid_dataset) == 19867

# Use the historical M3 configuration as the training reference.
reference_path = root / "runs/m3_seed0/config.json"
reference = json.loads(reference_path.read_text(encoding="utf-8"))

expected = {
    "seed": 0,
    "epochs": 10,
    "batch_size": 64,
    "num_workers": 2,
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "precision": "FP32",
    "initial_weights": "ResNet18_Weights.IMAGENET1K_V1",
    "f1_threshold": 0.5,
    "selection_metric": "validation_mAP",
    "selection_tie_rule": "earliest epoch",
    "epoch_seed_rule": "seed + epoch; epochs start at 1",
    "cudnn_deterministic": True,
    "cudnn_benchmark": False,
}
for key, value in expected.items():
    assert reference[key] == value, (key, reference[key], value)

assert train_dataset.attribute_names == reference["attribute_names"]
assert valid_dataset.attribute_names == reference["attribute_names"]

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    **{
        package: importlib.metadata.version(package)
        for package in (
            "torchvision", "numpy", "Pillow", "scikit-learn"
        )
    },
}
assert environment == reference["environment"], (
    "Environment differs from historical M3", environment
)

source_files = (
    "models/resnet18_geometry.py",
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "configs/attributes.yaml",
    "evaluation/attributes.py",
)
source_hashes = {}
for filename in source_files:
    data = (workspace / filename).read_bytes()
    digest = hashlib.sha256(data).hexdigest()
    assert digest == reference["source_sha256"][filename], filename
    source_hashes[filename] = digest

config = {
    **expected,
    "run_name": "oracle_seed0",
    "model": reference["model"],
    "attribute_names": reference["attribute_names"],
    "augmentation": reference["augmentation"],
    "geometry_mlp": reference["geometry_mlp"],
    "geometry_activation": reference["geometry_activation"],
    "fusion_features": reference["fusion_features"],
    "geometry_source": "CelebA aligned GT; shared preprocessing",
    "geometry_valid": "True for every train/validation sample",
    "failure_policy": "GT available; retain every sample",
    "all_parameters_trainable": True,
    "train_samples": 162770,
    "validation_samples": 19867,
    "source_sha256": source_hashes,
    "reference_m3_config_sha256": hashlib.sha256(
        reference_path.read_bytes()
    ).hexdigest(),
    "reference_m3_source_commit": reference["git_commit"],
    "environment": environment,
    "test_evaluated": False,
}

run_dir.mkdir(parents=True, exist_ok=True)


def save_json(path, value):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


config_path = run_dir / "config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == config, (
        "Existing Oracle configuration differs; preserved"
    )
else:
    assert not list(run_dir.glob("epoch_*.pt")), (
        "Checkpoints exist without a configuration; preserved"
    )
    save_json(config_path, config)

# Preserve the exact source files, including explanatory comments.
for filename in source_files:
    data = (workspace / filename).read_bytes()
    destination = run_dir / "source" / filename
    if destination.exists():
        assert destination.read_bytes() == data, filename
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

completed = [
    path for path in run_dir.glob("epoch_*.pt")
    if path.stem.removeprefix("epoch_").isdigit()
]
latest = max(
    completed,
    key=lambda path: int(path.stem.removeprefix("epoch_")),
    default=None,
)

seed_everything(config["seed"])
model = ResNet18GeometryAttributes(
    num_attributes=24,
    pretrained=(latest is None),
).to(device)

assert all(parameter.requires_grad for parameter in model.parameters())

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
criterion = nn.BCEWithLogitsLoss()

history = []
best = {"epoch": None, "map": None}
start_epoch = 1


def save_checkpoint(epoch):
    destination = run_dir / f"epoch_{epoch:02d}.pt"
    temporary = destination.with_name(destination.name + ".tmp")
    torch.save({
        "epoch": epoch,
        "config": config,
        "history": history,
        "best": best,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, temporary)
    os.replace(temporary, destination)


if latest is not None:
    state = torch.load(
        latest, map_location=device, weights_only=False
    )
    assert state["config"] == config
    assert state["epoch"] == int(latest.stem.removeprefix("epoch_"))
    model.load_state_dict(state["model_state_dict"], strict=True)
    optimizer.load_state_dict(state["optimizer_state_dict"])
    history = state["history"]
    best = state["best"]
    assert [row["epoch"] for row in history] == list(
        range(1, state["epoch"] + 1)
    )
    start_epoch = state["epoch"] + 1
    del state
    print("Resuming after:", latest.name, flush=True)
else:
    # Save the fresh initialization before any optimizer update.
    save_checkpoint(0)
    print("Fresh ImageNet initialization saved: epoch_00.pt", flush=True)


def make_loader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=shuffle,
        num_workers=config["num_workers"],
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
    )


def validate():
    model.eval()
    loss_sum = 0.0
    count = 0
    all_logits, all_targets, all_names = [], [], []

    with torch.inference_mode():
        for images, targets, points, names in make_loader(
            valid_dataset, shuffle=False
        ):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = torch.ones(
                images.shape[0], dtype=torch.bool, device=device
            )
            logits = model(images, points, valid)
            loss = criterion(logits, targets)

            assert torch.isfinite(logits).all().item()
            assert torch.isfinite(loss).item()

            count += images.shape[0]
            loss_sum += loss.item() * images.shape[0]
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            all_names.extend(names)

    assert count == 19867
    assert all_names == list(valid_dataset.names)
    logits = np.concatenate(all_logits)
    targets = np.concatenate(all_targets)
    metrics = evaluate_attributes(logits, targets, threshold=0.5)
    return loss_sum / count, metrics, logits, targets, all_names


print("Output:", run_dir, flush=True)
print("Geometry: GT; all samples valid", flush=True)
print("Training configuration verified against historical M3", flush=True)

for epoch in range(start_epoch, config["epochs"] + 1):
    seed_everything(config["seed"] + epoch)
    model.train()
    started = time.perf_counter()
    loss_sum = 0.0
    count = 0

    for batch_index, (images, targets, points, names) in enumerate(
        make_loader(train_dataset, shuffle=True), start=1
    ):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        points = points.to(device, non_blocking=True)
        valid = torch.ones(
            images.shape[0], dtype=torch.bool, device=device
        )

        optimizer.zero_grad(set_to_none=True)
        logits = model(images, points, valid)
        loss = criterion(logits, targets)
        assert torch.isfinite(loss).item()

        loss.backward()
        optimizer.step()

        count += images.shape[0]
        loss_sum += loss.item() * images.shape[0]

        if batch_index % 500 == 0:
            print(
                f"Epoch {epoch:02d}: {count}/{len(train_dataset)} "
                f"train BCE={loss_sum / count:.6f}",
                flush=True,
            )

    assert count == 162770
    valid_loss, metrics, _, _, _ = validate()

    # Strict improvement preserves the earlier epoch on an exact tie.
    if best["map"] is None or metrics["map"] > best["map"]:
        best = {"epoch": epoch, "map": metrics["map"]}

    row = {
        "epoch": epoch,
        "train_bce": loss_sum / count,
        "validation_bce": valid_loss,
        "validation_macro_f1_at_0.5": metrics["macro_f1"],
        "validation_map": metrics["map"],
        "seconds": time.perf_counter() - started,
    }
    history.append(row)
    save_checkpoint(epoch)
    save_json(run_dir / "history.json", history)
    save_json(run_dir / "best.json", best)

    print(
        f"Epoch {epoch:02d} saved | "
        f"train BCE={row['train_bce']:.6f} | "
        f"valid BCE={valid_loss:.6f} | "
        f"F1@0.5={metrics['macro_f1']:.6f} | "
        f"mAP={metrics['map']:.6f} | "
        f"{row['seconds'] / 60:.1f} min | "
        f"best epoch={best['epoch']}",
        flush=True,
    )

# Reload the selected checkpoint and save full clean-validation predictions.
assert len(history) == 10
selected_path = run_dir / f"epoch_{best['epoch']:02d}.pt"
selected = torch.load(
    selected_path, map_location=device, weights_only=False
)
assert selected["config"] == config
model.load_state_dict(selected["model_state_dict"], strict=True)
del selected

seed_everything(config["seed"] + best["epoch"])
valid_loss, metrics, logits, targets, names = validate()
assert abs(metrics["map"] - best["map"]) <= 1e-10

prediction_path = run_dir / "validation_predictions.npz"
temporary = prediction_path.with_name(prediction_path.name + ".tmp")
with temporary.open("wb") as handle:
    np.savez_compressed(
        handle,
        filenames=np.asarray(names),
        logits=logits,
        targets=targets,
        attribute_names=np.asarray(config["attribute_names"]),
        geometry_valid=np.ones(len(names), dtype=bool),
    )
os.replace(temporary, prediction_path)

save_json(run_dir / "validation_metrics.json", {
    **metrics,
    "validation_bce": valid_loss,
    "selected_epoch": best["epoch"],
    "selected_checkpoint": selected_path.name,
    "samples": len(names),
    "geometry_source": "GT",
    "threshold_fitting_performed": False,
    "test_evaluated": False,
})
save_json(run_dir / "history.json", history)
save_json(run_dir / "best.json", best)

print("\nSelected checkpoint:", selected_path)
print("Validation predictions:", prediction_path)
print(f"Clean validation F1@0.5: {metrics['macro_f1']:.6f}")
print(f"Clean validation mAP: {metrics['map']:.6f}")
print("Day 18 Oracle training and selected-checkpoint validation: PASS")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 145MB/s]


Fresh ImageNet initialization saved: epoch_00.pt
Output: /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed0
Geometry: GT; all samples valid
Training configuration verified against historical M3
Epoch 01: 32000/162770 train BCE=0.258512
Epoch 01: 64000/162770 train BCE=0.234138
Epoch 01: 96000/162770 train BCE=0.224097
Epoch 01: 128000/162770 train BCE=0.217779
Epoch 01: 160000/162770 train BCE=0.213646
Epoch 01 saved | train BCE=0.213322 | valid BCE=0.194073 | F1@0.5=0.734367 | mAP=0.813325 | 11.6 min | best epoch=1
Epoch 02: 32000/162770 train BCE=0.187493
Epoch 02: 64000/162770 train BCE=0.187208
Epoch 02: 96000/162770 train BCE=0.186727
Epoch 02: 128000/162770 train BCE=0.186933
Epoch 02: 160000/162770 train BCE=0.186826
Epoch 02 saved | train BCE=0.186787 | valid BCE=0.189859 | F1@0.5=0.729751 | mAP=0.815726 | 11.7 min | best epoch=2
Epoch 03: 32000/162770 train BCE=0.174496
Epoch 03: 64000/162770 train BCE=0.175059
Epoch 03: 96000/162770 train BCE=0.175211
Epoch 03: 1280

In [5]:
from pathlib import Path
import hashlib
import json

import numpy as np
from sklearn.metrics import f1_score

from evaluation.attributes import evaluate_attributes

root = Path("/content/drive/MyDrive/FacialVisualProfile")
run_dir = root / "runs/oracle_seed0"
output_dir = root / "runs/benchmark_preparation/day18"

config = json.loads((run_dir / "config.json").read_text())
history = json.loads((run_dir / "history.json").read_text())
best = json.loads((run_dir / "best.json").read_text())
saved_metrics = json.loads(
    (run_dir / "validation_metrics.json").read_text()
)

assert [row["epoch"] for row in history] == list(range(1, 11))

# max() retains the first occurrence when mAP is tied.
selected_row = max(history, key=lambda row: row["validation_map"])
assert best["epoch"] == selected_row["epoch"] == 3
assert best["map"] == selected_row["validation_map"]

checkpoint_path = run_dir / f"epoch_{best['epoch']:02d}.pt"
prediction_path = run_dir / "validation_predictions.npz"
assert checkpoint_path.is_file()

with np.load(prediction_path, allow_pickle=False) as data:
    filenames = data["filenames"].copy()
    logits = data["logits"].copy()
    targets = data["targets"].copy()
    attribute_names = data["attribute_names"].tolist()
    geometry_valid = data["geometry_valid"].copy()

expected_names = np.asarray([
    f"{index:06d}.jpg" for index in range(162771, 182638)
])

assert np.array_equal(filenames, expected_names)
assert logits.shape == targets.shape == (19867, 24)
assert np.isfinite(logits).all()
assert np.isin(targets, [0, 1]).all()
assert attribute_names == config["attribute_names"]
assert geometry_valid.dtype == np.bool_
assert geometry_valid.shape == (19867,)
assert geometry_valid.all()
assert all(np.unique(targets[:, i]).size == 2 for i in range(24))

# Independently check alignment against historical R18 predictions.
r18_prediction_path = (
    root / "runs/r18_seed0/day7/validation_predictions.npz"
)
with np.load(r18_prediction_path, allow_pickle=False) as reference:
    assert np.array_equal(filenames, reference["filenames"])
    assert np.array_equal(targets, reference["targets"])
    assert attribute_names == reference["attribute_names"].tolist()

baseline = evaluate_attributes(logits, targets, threshold=0.5)
assert saved_metrics["selected_epoch"] == best["epoch"]
assert saved_metrics["selected_checkpoint"] == checkpoint_path.name

for key in ("macro_f1", "map"):
    assert abs(baseline[key] - saved_metrics[key]) <= 1e-10
assert abs(baseline["map"] - best["map"]) <= 1e-10

print("Full validation IDs, targets, GT validity and metrics: PASS")

# Stable float64 sigmoid, matching the historical evaluation.
values = logits.astype(np.float64)
probabilities = np.empty_like(values)
positive = values >= 0
probabilities[positive] = 1 / (1 + np.exp(-values[positive]))
exp_values = np.exp(values[~positive])
probabilities[~positive] = exp_values / (1 + exp_values)

grid_integers = list(range(5, 96))
grid = [value / 100 for value in grid_integers]
thresholds = []
tuned_f1 = []

for attribute_index in range(24):
    scores = [
        float(f1_score(
            targets[:, attribute_index],
            probabilities[:, attribute_index] >= threshold,
            zero_division=0,
        ))
        for threshold in grid
    ]
    selected_index = min(
        range(len(grid)),
        key=lambda index: (
            -scores[index],
            abs(grid_integers[index] - 50),
            grid_integers[index],
        ),
    )
    thresholds.append(grid[selected_index])
    tuned_f1.append(scores[selected_index])


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


print("Computing checkpoint and prediction SHA256...")

report = {
    "run_name": "oracle_seed0",
    "checkpoint": checkpoint_path.name,
    "selected_epoch": best["epoch"],
    "checkpoint_sha256": file_sha256(checkpoint_path),
    "prediction_path": str(prediction_path),
    "prediction_sha256": file_sha256(prediction_path),
    "fit_split": "valid",
    "samples": len(filenames),
    "attribute_names": attribute_names,
    "thresholds": thresholds,
    "objective": "per_attribute_F1",
    "grid": grid,
    "tie_break": "closest_to_0.5_then_lower",
    "comparison": ">=",
    "baseline_macro_f1": baseline["macro_f1"],
    "tuned_macro_f1": float(np.mean(tuned_f1)),
    "map": baseline["map"],
    "per_attribute_f1_at_0.5": baseline["per_attribute_f1"],
    "per_attribute_f1_tuned": tuned_f1,
    "per_attribute_ap": baseline["per_attribute_ap"],
    "geometry_pipeline": "CelebA aligned GT; shared preprocessing",
    "masked_validation_samples": 0,
    "config_sha256": file_sha256(run_dir / "config.json"),
    "source_sha256": config["source_sha256"],
    "environment": config["environment"],
    "test_evaluated": False,
    "note": (
        "Thresholds fitted on clean validation only. Tuned validation F1 "
        "is an in-sample fitting result, not independent evaluation. "
        "Reuse these thresholds unchanged for clean/corrupted test inputs."
    ),
}

output_dir.mkdir(parents=True, exist_ok=True)
destination = output_dir / "oracle_seed0.json"
payload = json.dumps(report, indent=2, allow_nan=False) + "\n"

if destination.exists():
    assert destination.read_text() == payload, (
        "Existing threshold report differs; preserved"
    )
else:
    destination.write_text(payload, encoding="utf-8")

print(f"Oracle F1@0.5: {baseline['macro_f1']:.6f}")
print(f"Oracle tuned validation F1: {np.mean(tuned_f1):.6f}")
print(f"Oracle mAP: {baseline['map']:.6f}")
print("Saved:", destination)
print("Day 18 Oracle clean-validation thresholds: PASS")

Full validation IDs, targets, GT validity and metrics: PASS
Computing checkpoint and prediction SHA256...
Oracle F1@0.5: 0.741068
Oracle tuned validation F1: 0.764027
Oracle mAP: 0.818460
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/benchmark_preparation/day18/oracle_seed0.json
Day 18 Oracle clean-validation thresholds: PASS


In [6]:
from pathlib import Path
import hashlib
import json
import zipfile

from google.colab import files

root = Path("/content/drive/MyDrive/FacialVisualProfile")
run_dir = root / "runs/oracle_seed0"
threshold_path = (
    root / "runs/benchmark_preparation/day18/oracle_seed0.json"
)

threshold = json.loads(threshold_path.read_text())
best = json.loads((run_dir / "best.json").read_text())
history = json.loads((run_dir / "history.json").read_text())
metrics = json.loads((run_dir / "validation_metrics.json").read_text())

assert len(history) == 10
assert best["epoch"] == metrics["selected_epoch"] == 3
assert threshold["checkpoint"] == metrics["selected_checkpoint"]
assert threshold["samples"] == 19867
assert threshold["masked_validation_samples"] == 0
assert threshold["test_evaluated"] is False
assert len(threshold["thresholds"]) == 24
assert threshold["grid"] == [i / 100 for i in range(5, 96)]
assert threshold["tie_break"] == "closest_to_0.5_then_lower"

def sha256(data):
    return hashlib.sha256(data).hexdigest()

payloads = {}

for filename in (
    "config.json",
    "history.json",
    "best.json",
    "validation_metrics.json",
):
    payloads[
        f"docs/results/oracle_training/day18/{filename}"
    ] = (run_dir / filename).read_bytes()

threshold_bytes = threshold_path.read_bytes()
payloads[
    "docs/results/benchmark_preparation/day18/oracle_seed0.json"
] = threshold_bytes
payloads[
    "configs/thresholds/oracle_seed0.json"
] = threshold_bytes

assert sha256(payloads[
    "docs/results/oracle_training/day18/config.json"
]) == threshold["config_sha256"]

manifest = {
    "run_name": "oracle_seed0",
    "selected_checkpoint": threshold["checkpoint"],
    "checkpoint_sha256": threshold["checkpoint_sha256"],
    "prediction_sha256": threshold["prediction_sha256"],
    "payload_sha256": {
        name: sha256(data) for name, data in payloads.items()
    },
    "test_evaluated": False,
    "note": (
        "JSON archive only. Checkpoints and validation predictions "
        "remain in Drive. Day 17 benchmark freeze is not modified."
    ),
}
payloads[
    "docs/results/oracle_training/day18/archive_manifest.json"
] = (
    json.dumps(manifest, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

destination = Path("/content/day18_oracle_archive.zip")
with zipfile.ZipFile(
    destination, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name, data in payloads.items():
        archive.writestr(name, data)

with zipfile.ZipFile(destination) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(payloads)

print("Archive:", destination)
print("Files:", len(payloads))
print("Day 18 Oracle archive preparation: PASS")
files.download(str(destination))

Archive: /content/day18_oracle_archive.zip
Files: 7
Day 18 Oracle archive preparation: PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>